# 04 · NASA Rotor 67 rebuilt with BladeGen

The 14 blade sections and the flowpath of NASA Rotor 67 (Strazisar et al., NASA TP-2879, 1989; tables in
`data/rotor67`) are

1. **constructed as published** (surface splines with the tabulated end angles, edge circles, tangency points —
   the C++ port of `rotor67_sections.py`),
2. **inverse-designed** section by section: every section is brought to the unstaggered frame of its stream
   surface and the BladeGen profile parameters are fitted (C++, in parallel threads),
3. **rebuilt in 3-D**: a C2 NURBS skin through the 14 fitted sections, trimmed on the hub and on the casing
   contour of surface 14 offset by the 1.016 mm tip clearance (TP-2879), and
4. **filleted** at the hub with the TP-2879 radius of 1.78 mm (button below).

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# the bladegen package lives in <build>/python (cmake --preset python); BLADEGEN_PY overrides
REPO = Path.cwd().resolve().parents[1]
for cand in [os.environ.get("BLADEGEN_PY", ""), REPO / "build" / "python" / "python"]:
    if cand and Path(cand).is_dir() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))
import bladegen as bg
import bladegen.notebook as nb
OUT = Path(os.environ.get("BLADEGEN_TEST_OUTPUT", REPO / "build" / "notebook_output"))
OUT.mkdir(parents=True, exist_ok=True)
print("BladeGen", bg.__version__, "| widgets:", nb.HAVE_WIDGETS)

In [ ]:
data = bg.read_rotor67(str(REPO / "data" / "rotor67" / "Rotor_14_Sections_and_Flowpath_EN.txt"))
print(f"{len(data.sections)} sections, {data.n_blades} blades, tip clearance {1e3 * data.tip_clearance:.4f} mm")
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(*data.hub.T * 1000, "k", lw=1.5); ax[0].plot(*data.shroud.T * 1000, "k", lw=1.5)
for s in data.sections:
    ax[0].plot(*s.streamline.sample(60).T * 1000, color="0.5", lw=0.8)
ax[0].set_aspect("equal"); ax[0].set_xlim(-30, 120); ax[0].grid(alpha=.3); ax[0].set_title("flowpath and section stream lines [mm]")
for s in data.sections:
    ax[1].plot(*s.profile_m_theta().T, lw=0.9)
ax[1].set_xlabel("m [m]"); ax[1].set_ylabel("θ [rad]"); ax[1].grid(alpha=.3); ax[1].set_title("published sections in (m, θ)")
plt.show()

## Fit all 14 sections (C++, threads)

The fit releases the GIL, so the 14 sections run in parallel. `fairness_weight` keeps the fitted edges free of
S-bends. The table reports the 2-D deviation (% chord) and the 3-D deviation of the published points from the
rebuilt sections (µm).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def fit(k):
    s = data.sections[k]
    opt = bg.FitOptions(); opt.fairness_weight = 0.3
    return bg.fit_section(s.profile_conformal(), s.streamline, k / 13.0, bg.ProfileParams.cda_start(), opt)
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
    fits = list(ex.map(fit, range(len(data.sections))))
print(f"14 sections fitted in {time.perf_counter() - t0:.0f} s\n")
print(" sec  stagger  RMS 2D [%c]  max 2D [%c]   RMS 3D [um]  max 3D [um]   quality")
for s, f in zip(data.sections, fits):
    q = bg.check_quality(bg.Profile2D(f["fit"]["params"]))
    print(f" {s.index:3d}  {np.degrees(f['frame']['stagger']):6.2f}   {100 * f['fit']['rms']:9.4f}   {100 * f['fit']['max']:9.4f}"
          f"   {1e6 * f['rms_3d']:10.2f}  {1e6 * f['max_3d']:10.2f}   {q['errors']} err / {q['warnings']} warn")

## Published vs rebuilt sections (hub, mid, tip) and their curvature

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for row, k in enumerate([0, 6, 13]):
    f = fits[k]; prof = bg.Profile2D(f["fit"]["params"]); q = bg.check_quality(prof)
    fr = f["frame"]
    a = axes[row, 0]; nb.draw_profile(a, prof, q, control=False, points=False)
    a.plot(*fr["ss"].T, "o", mfc="none", mec="k", ms=2.5); a.plot(*fr["ps"].T, "o", mfc="none", mec="k", ms=2.5)
    nb._badge(a, q); a.set_title(f"section {data.sections[k].index}: published (o) vs BladeGen")
    a = axes[row, 1]; nb.draw_profile(a, prof, q, control=False, points=False)
    a.plot(*fr["ss"].T, "o", mfc="none", mec="k", ms=4); a.plot(*fr["ps"].T, "o", mfc="none", mec="k", ms=4)
    r = f["fit"]["params"].le.radius; a.set_xlim(-2 * r, 8 * r); a.set_ylim(-5 * r, 5 * r); a.set_title("LE zoom")
    nb.draw_curvature(axes[row, 2], q, prof)
plt.tight_layout(); plt.show()

## The 3-D blade: C2 NURBS skin through the 14 fitted sections

The definition is written to JSON (exact B-spline stream lines) so the CLI (`bladegen build`) reproduces the
same blade.

In [ ]:
definition = bg.BladeDefinition()
definition.name = "NASA_Rotor_67"; definition.n_blades = data.n_blades; definition.tip_clearance = data.tip_clearance
definition.stations = [f["station"] for f in fits]
definition.channel = bg.MeridionalChannel(data.hub_curve(), data.shroud_curve())
bg.write_blade_json(definition, str(OUT / "rotor67_bladegen.json"))
blade = bg.Blade(definition)
loops = [s["xyz"] for s in blade.sections(60, 200)]
inside = None
mid = blade.section(0.5, 60, 200)["xyz"]; inside = 0.5 * (mid[len(mid) // 4] + mid[3 * len(mid) // 4])
t0 = time.perf_counter()
shapes = bg.build_shapes(loops, data.hub_curve(), data.sections[-1].streamline, inside, data.tip_clearance, 0.0, 1.0)
print(f"skin + solid in {time.perf_counter() - t0:.1f} s: valid {shapes.blade_valid}, volume {1e6 * shapes.blade_volume:.3f} cm³")
S = shapes.surface
print(f"skin degree ({S.u_degree},{S.v_degree}), poles {S.n_poles}, seam jumps (gap, d1, d2) = {tuple(f'{x:.1e}' for x in S.seam_jumps(0.5))}")
pub = np.vstack([f["published_xyz"][::6] for f in fits])
d = shapes.blade_face.distances(pub)
print(f"published points → NURBS skin: RMS {1e6 * np.sqrt(np.mean(d ** 2)):.2f} um, max {1e6 * d.max():.2f} um ({len(d)} points)")

In [ ]:
fig = nb.show_shapes([(shapes.blade, "lightsteelblue", "Rotor 67 blade (BladeGen)"), (shapes.hub, "0.8", "hub sector")],
                     title="NASA Rotor 67 — C2 NURBS blade, tip clearance 1.016 mm")
try:
    import plotly.graph_objects as go
    fig.add_trace(go.Scatter3d(x=pub[:, 0] * 1000, y=pub[:, 1] * 1000, z=pub[:, 2] * 1000, mode="markers",
                               marker=dict(size=1.5, color="black"), name="published points"))
    fig.show()
except ImportError:
    plt.show()

## Hub fillet, r = 1.78 mm (TP-2879) — and export

Press **Add hub fillet**. The rolling-ball construction reports its own checks: the ball touches both walls at
exactly r, and the fillet is tangent to both (G1 angle).

In [ ]:
sf = bg.build_shapes(loops, data.hub_curve(), data.sections[-1].streamline, inside, data.tip_clearance, 1.78e-3, 1.0)
print(sf.fillet_message, f"| G1 {sf.fillet_g1_deg:.3f}°, ball error {sf.fillet_ball_error:.1e} m")
if sf.fillet_ok:
    sf.blade_on_hub.write(str(OUT / "rotor67_hub_blade_fillet.step"))
    sf.blade_with_fillet.write(str(OUT / "rotor67_blade_fillet.step"))
shapes.blade.write(str(OUT / "rotor67_blade.step"))
fig = nb.show_shapes([(sf.blade_on_hub if sf.fillet_ok else shapes.blade, "lightsteelblue", "hub + blade + fillet")],
                     title="Rotor 67 with the 1.78 mm hub fillet")
try:
    fig.show()
except Exception:
    plt.show()
if nb.HAVE_WIDGETS:
    import ipywidgets as W
    from IPython.display import display
    radius = W.BoundedFloatText(value=1.78, min=0.2, max=6.0, step=0.01, description="fillet [mm]")
    btn = W.Button(description="Add hub fillet", button_style="primary", icon="magic")
    msg = W.HTML(); view = W.Output()
    def add(_):
        msg.value = "<i>building ...</i>"
        s2 = bg.build_shapes(loops, data.hub_curve(), data.sections[-1].streamline, inside, data.tip_clearance,
                             radius.value * 1e-3, 1.0)
        msg.value = (f"<b style='color:{'green' if s2.fillet_ok else 'red'}'>{s2.fillet_message}</b> "
                     f"G1 {s2.fillet_g1_deg:.3f}°")
        if s2.fillet_ok:
            s2.blade_on_hub.write(str(OUT / f"rotor67_hub_blade_fillet_{radius.value:.2f}mm.step"))
            with view:
                view.clear_output(wait=True)
                nb.show_shapes([(s2.blade_on_hub, "lightsteelblue", "hub + blade + fillet")]).show()
    btn.on_click(add)
    display(W.VBox([W.HBox([radius, btn]), msg, view]))

## Compare with the STEP files of the Python scripts (if present)

`newblade/rotor67/rotor67_blade.step` was written by `rotor67_passage_solid.py` (blade to the casing, no tip
clearance by default); volumes are compared for orientation only.

In [ ]:
py_step = Path(os.environ.get("ROTOR67_PY_STEP", r"C:\Users\ASUS\Documents\newblade\rotor67\rotor67_blade.step"))
if py_step.is_file():
    py = bg.read_step(str(py_step))
    print(f"Python blade (no tip clearance): {1e6 * py.volume:.3f} cm³   BladeGen (tip clearance 1.016 mm): {1e6 * shapes.blade_volume:.3f} cm³")
else:
    print("Python STEP not found; skipped")